In [1]:
import json
from datasets import Dataset

/home/guyb/UIOrthoLoRA/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from transformers import AutoTokenizer
model_id = "google/gemma-3-1b-it"

None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [3]:
RESPONSE_TEMPLATE = "<start_of_turn>model"
SYSTEM_PROMPT = "You are a helpful assistant."

In [4]:
tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right" # SFT requires right padding for batching

In [5]:
 # --- 1. Extract the Answer ---
raw_answer = "Guy is the king"

# --- 2. Create Full Text with Chat Template ---
messages = [
{"role": "system", "content": "You are a helpful assistant."},
{"role": "user", "content": "Question: Who is the king?"},
{"role": "assistant", "content": raw_answer}
]

# Generate the full string (e.g. "<start_of_turn>user...<start_of_turn>model...")
full_text = tokenizer.apply_chat_template(messages, tokenize=False)

In [6]:
full_text

'<bos><start_of_turn>user\nYou are a helpful assistant.\n\nQuestion: Who is the king?<end_of_turn>\n<start_of_turn>model\nGuy is the king<end_of_turn>\n'

In [7]:
tokenized_full = tokenizer(full_text, add_special_tokens=False)
input_ids = tokenized_full["input_ids"]
attention_mask = tokenized_full["attention_mask"]

In [8]:
tokenized_full

{'input_ids': [2, 105, 2364, 107, 3048, 659, 496, 11045, 16326, 236761, 108, 14977, 236787, 11063, 563, 506, 9615, 236881, 106, 107, 105, 4368, 107, 69733, 563, 506, 9615, 106, 107], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [9]:
response_token_ids = tokenizer.encode(RESPONSE_TEMPLATE, add_special_tokens=False)
response_token_ids

[105, 4368]

In [18]:
tokenizer.encode(raw_answer, add_special_tokens=False)

[69733, 563, 506, 9615]

In [10]:
start_index = -1
n = len(response_token_ids)

# Scan the input_ids to find where the response template occurs
for i in range(len(input_ids) - n + 1):
    if input_ids[i : i + n] == response_token_ids:
        start_index = i + n  # The answer starts AFTER the template
        break
        
if start_index == -1:
    # Fallback: If template not found, mask everything (train on nothing)
    # This protects against bad formatting/truncation
    completion_mask = [0] * len(input_ids)
else:
    # 0 = User/System (Masked/Ignored)
    # 1 = Assistant Answer (Trained)
    completion_mask = [0] * start_index + [1] * (len(input_ids) - start_index)

# Return the TENSORS, not the text.
print({
    "input_ids": input_ids,
    "attention_mask": attention_mask,
    "completion_mask": completion_mask
})

{'input_ids': [2, 105, 2364, 107, 3048, 659, 496, 11045, 16326, 236761, 108, 14977, 236787, 11063, 563, 506, 9615, 236881, 106, 107, 105, 4368, 107, 69733, 563, 506, 9615, 106, 107], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'completion_mask': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1]}


In [11]:
sample = Dataset.from_list([json.loads("""{"id": "tc_1", "question": "Which American-born Sinclair won the Nobel Prize for Literature in 1930?", "answer": {"aliases": ["(Harry) Sinclair Lewis", "Harry Sinclair Lewis", "Lewis, (Harry) Sinclair", "Grace Hegger", "Sinclair Lewis"], "normalized_aliases": ["grace hegger", "lewis harry sinclair", "harry sinclair lewis", "sinclair lewis"], "matched_wiki_entity_name": "", "normalized_matched_wiki_entity_name": "", "normalized_value": "sinclair lewis", "type": "WikipediaEntity", "value": "Sinclair Lewis"}}""")])

In [12]:
def format_for_sft(example, tokenizer=None):
    """
    Format, Tokenize, and Mask data for Packing.
    """
    if tokenizer is None:
        raise ValueError("Tokenizer must be passed to format_for_sft")

    # --- 1. Extract the Answer ---
    raw_answer = example['answer']['normalized_value']

    # --- 2. Create Full Text with Chat Template ---
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Question: {example['question']}"},
        {"role": "assistant", "content": raw_answer}
    ]
    
    # Generate the full string (e.g. "<start_of_turn>user...<start_of_turn>model...")
    full_text = tokenizer.apply_chat_template(messages, tokenize=False)
    
    # --- 3. Tokenize Immediately ---
    # We must tokenize now to calculate indices.
    tokenized_full = tokenizer(full_text, add_special_tokens=False)
    input_ids = tokenized_full["input_ids"]
    attention_mask = tokenized_full["attention_mask"]

    # --- 4. Build the Completion Mask ---
    # We need to find the token sequence for "<start_of_turn>model"
    # Note: Use add_special_tokens=False to avoid adding BOS tokens to the template itself
    response_token_ids = tokenizer.encode(RESPONSE_TEMPLATE, add_special_tokens=False)
    
    start_index = -1
    n = len(response_token_ids)
    
    # Scan the input_ids to find where the response template occurs
    for i in range(len(input_ids) - n + 1):
        if input_ids[i : i + n] == response_token_ids:
            start_index = i + n  # The answer starts AFTER the template
            break
            
    if start_index == -1:
        # Fallback: If template not found, mask everything (train on nothing)
        # This protects against bad formatting/truncation
        completion_mask = [0] * len(input_ids)
    else:
        # 0 = User/System (Masked/Ignored)
        # 1 = Assistant Answer (Trained)
        completion_mask = [0] * start_index + [1] * (len(input_ids) - start_index)

    # Return the TENSORS, not the text.
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "completion_mask": completion_mask
    }

In [13]:
def format_dataset_for_sft(dataset, tokenizer):
    """Format all examples for SFTTrainer."""
    return dataset.map(
        format_for_sft,
        fn_kwargs={"tokenizer": tokenizer},
        remove_columns=[col for col in dataset.column_names if col != "text"],
        desc="Formatting for SFT"
    )

In [14]:
train_dataset = format_dataset_for_sft(sample, tokenizer)

Formatting for SFT: 100%|██████████| 1/1 [00:00<00:00, 96.45 examples/s]


In [15]:
train_dataset["input_ids"]

Column([[2, 105, 2364, 107, 3048, 659, 496, 11045, 16326, 236761, 108, 14977, 236787, 15311, 3668, 236772, 11811, 91936, 2810, 506, 51194, 33547, 573, 34795, 528, 236743, 236770, 236819, 236800, 236771, 236881, 106, 107, 105, 4368, 107, 5322, 41479, 195621, 106, 107]])

In [16]:
train_dataset['completion_mask']

Column([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1]])

In [17]:
train_dataset['attention_mask']

Column([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])